# MeteoScreening `SWC_FF1_0.3_1` (2020-2025) from database (influxdb)

***
**Site**: CH-LAE &nbsp;&nbsp;|&nbsp;&nbsp; **Variable**: `SWC_FF1_0.3_1` &nbsp;&nbsp;|&nbsp;&nbsp; **Period**: 2020-2025  
**Cross-check**: the other depths of the same soil profile (`0.05`, `0.1`, `0.2`, `0.5` m) and the screened precipitation product — see *Cross-check* below  
**Derived from**: the sibling notebook `SWC_FF1_0.05_1_2020-2025.ipynb` (itself derived from the diive template `DatabaseInfluxStepwiseMeteoScreening.ipynb`)  
**Author**: Lukas Hörtnagl (holukas@ethz.ch)

## ℹ️ About this notebook
Download raw soil water content from the InfluxDB database, quality-screen and correct it on the **high-resolution** data, resample to 30MIN, and upload the result back to the database. Screening uses `StepwiseMeteoScreeningDb` from [diive](https://github.com/holukas/diive) (`diive/preprocessing/qaqc/meteoscreening.py`); download and upload use diive's in-house InfluxDB engine (`InfluxIO`, in `diive/core/io/db/influx`).

**Flow:** download (`InfluxIO`) → cross-check against the rest of the soil profile → screen on high-res data (`diive`) → resample to 30MIN → validate against the profile → upload.

**Outlier detection is stepwise:** run a test, inspect its preview plot, then commit it with `mscr.addflag()`. Re-run with different parameters as often as you like before committing. Run only the tests a variable actually needs. At the end all committed flags are aggregated into one overall quality flag `QCF`.

> ### 🔀 This series is not homogeneous: it holds two sensors
>
> **`SWC_FF1_0.3_1` is the one depth of the FF1 profile where the field name spans a sensor change.** The original 30 cm probe failed in April 2024 — and in failing it pulled the whole FF1 SDI-12 bus down, which is why *every* depth has a coverage gap in April 2024. It was unplugged and, on **4 Mar 2025**, replaced. The replacement could not go back in the old hole, so it went in about 40 cm **down the slope**, in genuinely different soil. So this record is really **two sensors in two places**: an *old-probe era* (2020-04-10 → 2024-04-11) and a *new-probe era* (2025-03-04 → present), separated by a **327-day dead period**.
>
> **How it is handled here (settled, not a hedge).** Both eras are screened under the one field name and **uploaded as measured, with no level adjustment**. The break at 2025-03-04 is documented prominently (this note plus the dedicated *The sensor swap* section below) so downstream users know the series steps there. The measured level offset is **about +3 % VWC** (new probe reads higher relative to the profile; see the swap section for the method) — it is reported as evidence, **not** applied. An offset correction would write values the sensor never measured; the ~3 % is confounded by season and by a genuinely different soil location; and a documented break is honest where a silent homogenisation is not. The swap date lives in one place, the constant `SENSOR_SWAP`, so the decision is easy to find and easy to change.

**This notebook is a copy of the `0.05` pattern**, re-derived for the 30 cm sensor. `REMOVE_DATES`, the candidate list and the closing prose are all this depth's own — the 5 cm removal window has been deleted, because it belongs to that probe alone.

> **Database access** needs the `influxdb-client` package, which this project pulls in via the **`diive[db]` extra** (declared in `pyproject.toml`). Note that diive *also* ships a `db` dependency group, but dependency groups are local to the project that declares them — `uv sync --group db` only works inside the diive repo, not from here. From this repo the extra is the only route.

## ⏱️ Timestamp convention (important)
The database stores every timestamp in **UTC** and as **`TIMESTAMP_END`** (the stamp marks the *end* of the averaging interval). Getting the timestamp right is the one thing that must not go wrong, because the day/night split during screening — and the value written back to the DB — both depend on it.

The single knob is `TIMEZONE_OFFSET_TO_UTC_HOURS` (set in *User settings*). It is applied **identically** on download and on upload. Here is the full round-trip, all handled for you:

| Stage | Timezone | Convention | Done by |
|---|---|---|---|
| Database | UTC | `TIMESTAMP_END` | InfluxDB |
| After `dbc.download(..., timezone_offset_to_utc_hours=N)` | local (UTC+N) | `TIMESTAMP_END` | InfluxIO |
| During screening | local | `TIMESTAMP_MIDDLE` (converted internally) | `StepwiseMeteoScreeningDb` |
| After `mscr.resample()` | local | back to `TIMESTAMP_END` | diive |
| After `dbc.upload_singlevar(..., timezone_offset_to_utc_hours=N)` | UTC | `TIMESTAMP_END` | InfluxIO |

So screening runs on correct local middle-of-period timestamps, and the resampled value lands on the correct UTC end-of-period stamp in the DB. **`TIMEZONE_OFFSET_TO_UTC_HOURS` must match the timezone the raw data was logged in** (e.g. `1` for CET winter time) and must be the same value everywhere. This notebook prints the timestamps right after download and right after the verification download so you can confirm they look correct.

Everything in the *Cross-check* section works on `TIMESTAMP_END` as downloaded. The precipitation product read there is stored on `TIMESTAMP_MID` and is shifted to `TIMESTAMP_END` before use.

## ✏️ User settings (please adjust)

Adjust these before running. What each setting means:

**Site**
- `SITE`, `SITE_LAT`, `SITE_LON`: site ID and coordinates. The coordinates set the day/night split used during screening (not used for soil moisture — see *Outlier detection*).

**Variable to screen**
- `PROFILE`, `DEPTH`, `REPL`: the sensor's position tags. `FIELD` is assembled from them and is the InfluxDB `_field`. **This is the only place to change when screening another depth.**
- `MEASUREMENT`: exactly **one** measurement grouping the variables — `SWC` for soil water content.
- `COMPANION_DEPTHS`: the other depths of the same profile, used as the cross-check reference. They are *not* screened here.

**Time range to screen**
- `START`: first timestamp to screen — **is** included.
- `STOP`: upper bound — **is not** included.

**Data settings**
- `TIMEZONE_OFFSET_TO_UTC_HOURS`: the critical timestamp knob — see the *Timestamp convention* section above. Must match how the raw data was logged (e.g. `1` for CET winter time).
- `DATA_VERSION`: the source data version in the database (`raw`).
- `DIRCONF`: local folder holding the database connection config.

**Resampling**
- `RESAMPLING_FREQ`: the screened high-res data is resampled to this frequency.
- `RESAMPLING_AGG`: aggregation used when resampling. Soil water content is a **state**, so `'mean'` — never `'sum'`.

**Cross-check**
- `PREC_PRODUCT` / `PREC_COL`: the screened precipitation product, used to test whether the sensor still *responds* to rain. Produced by [`30_PRODUCTS/08_METEO_PREC_2004-2025.ipynb`](../../30_PRODUCTS/08_METEO_PREC_2004-2025.ipynb).
- `FIELDBOOK`: the GIN fieldbook export, read directly here so that every candidate period is adjudicated against the site record in the notebook rather than by hand.

**Parameter help**
- `SHOW_PARAM_HELP`: set `True` to print the full docstring (all parameters) of each screening method right before it runs. Leave `False` for a clean, concise notebook.

In [ ]:
# --- Site ---
SITE = 'ch-lae'
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389  # CH-LAE

# --- Variable to screen ---
# Verified 21 Jul 2026 against measurement SWC of bucket ch-lae_raw: the FF1 profile
# holds five depths (0.05, 0.1, 0.2, 0.3, 0.5), all TEROS 12, all starting
# 2020-04-10. (The older M5 profile, SWC_M5_*, ends the same day and is a
# different, now-trashed sensor set - see the closing note. The FF3 profile
# starts only in 2026 and is out of scope here.)
PROFILE = 'FF1'  # horizontal position (soil profile)
DEPTH = '0.3'  # <-- the only setting to change when screening another depth
REPL = '1'  # replicate
FIELD = f'SWC_{PROFILE}_{DEPTH}_{REPL}'
FIELDS = [FIELD]  # StepwiseMeteoScreeningDb expects a list
MEASUREMENT = 'SWC'

# (!) THE SENSOR SWAP. This depth's field name spans two physical sensors: the
# original 30 cm probe (to 2024-04-11) and its replacement 40 cm down the slope
# (from 2025-03-04), with a 327-day dead period between them. The date below is
# established from the DATA in 'The sensor swap' section (the fieldbook says the
# old probe was unplugged 23 Apr 2024, but it went silent on 2024-04-11, ~12 days
# earlier). Every era-split diagnostic reads this one constant - change it here,
# nowhere else. Both eras are uploaded as measured; no offset is applied.
SENSOR_SWAP = '2025-03-04'  # first record of the replacement probe (from the data)
OLD_PROBE_LAST = '2024-04-11 01:46:00'  # last record of the original probe (from the data)

# The rest of the profile: the cross-check reference. Screened by their own notebooks.
COMPANION_DEPTHS = ['0.05', '0.1', '0.2', '0.3', '0.5']
COMPANION_FIELDS = [f'SWC_{PROFILE}_{d}_{REPL}' for d in COMPANION_DEPTHS if d != DEPTH]

# 0.05 lost contact with the soil 2020-08-01 -> 2020-10-24 (documented in its own
# notebook). It is a companion here, and that window is still in the raw cache, so it
# is masked out of the companion mean below or it would distort this depth's
# cross-check. This is the only cross-depth window carried in; it is NOT a fault of
# this depth.
COMPANION_MASK = {'SWC_FF1_0.05_1': ('2020-08-01', '2020-10-24')}

# --- Time range to screen ---
# Raw resolution is 10MIN until the 2021 logger rebuild and 1MIN after, so this
# range is ~2 million records (~3 min to download). The profile starts
# 2020-04-10; asking for 2020-01-01 simply starts at the first record.
START = '2020-01-01 00:00:01'  # included
STOP = '2026-01-01 00:00:01'  # not included

# --- Data settings ---
DATA_VERSION = 'raw'
TIMEZONE_OFFSET_TO_UTC_HOURS = 1  # UTC+01:00 (CET, winter time). Must match how the raw data was logged.
DIRCONF = r'F:\dev\poet\configs'
# DIRCONF = r'P:\Flux\RDS_calculations\_scripts\_configs\configs'

# --- Resampling ---
RESAMPLING_FREQ = '30min'  # screened high-res data is resampled to this frequency
RESAMPLING_AGG = 'mean'  # (!) soil water content is a state variable, never summed

# --- Cross-check: the screened precipitation product (30_PRODUCTS/08) ---
PREC_PRODUCT = (r'F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data'
                r'\workflow\10_METEO\30_PRODUCTS'
                r'\08_METEO_PREC_GAPFILLED_2004-2025.parquet')
PREC_COL = 'PREC_TOT_T1_47_1'

# --- Cross-check: the GIN fieldbook export (site record) ---
FIELDBOOK = (r'F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data'
             r'\fieldbook_gin\CH-LAE-laegeren-export_20260719.csv')

# --- Physical range of soil water content at this site, in % (volumetric) ---
# Used by the absolute-limits test and by the candidate scan. 0 is the hard
# physical floor; 60 sits above any plausible porosity for this soil and is
# far above the highest value ever measured at this depth (36.5%, either era).
SWC_MIN, SWC_MAX = 0, 60

# --- Parameter help ---
SHOW_PARAM_HELP = False

## 🤖 Auto settings

### Buckets (do not adjust)

In [ ]:
BUCKET_RAW = f'{SITE}_raw'  # source bucket, e.g. 'ch-lae_raw'
BUCKET_PROCESSED = f'{SITE}_processed'  # destination bucket, e.g. 'ch-lae_processed'
print(f'Screening variable:            {FIELD}')
print(f'Cross-check against:           {COMPANION_FIELDS}')
print(f'Source bucket (raw data):      {BUCKET_RAW}')
print(f'Destination bucket (processed): {BUCKET_PROCESSED}')

### Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import diive as dv
from diive.core.io.db.influx import InfluxIO  # needs influxdb-client, via the diive[db] extra
from diive.core.times.times import detect_freq_groups  # the same grouping the screening uses internally

warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
pd.set_option('display.max_rows', 30)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"diive v{importlib.metadata.version('diive')}")

## ⬇️ Download data from database

### Connect to database

In [ ]:
dbc = InfluxIO(dirconf=DIRCONF)

Optional — list all fields available in the measurement (does not check the selected time range):

In [ ]:
display(dbc.show_fields_in_measurement(bucket=BUCKET_RAW, measurement=MEASUREMENT))

### Download
Returns three objects:
- `data_simple`: high-res time series, one column per variable (nice to look at).
- `data_detailed`: dict `{varname: DataFrame}` with each variable's time series **and its database tags** — this is what the screening consumes.
- `assigned_measurements`: the auto-detected measurement per variable (a sanity check).

In [ ]:
%%time
data_simple, data_detailed, assigned_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

### Inspect downloaded data

In [ ]:
data_simple

In [ ]:
assigned_measurements

Drop any requested variable that has no data in this period:

In [ ]:
vars_not_available = [v for v in FIELDS if v not in data_detailed.keys()]
for rem in vars_not_available:
    FIELDS.remove(rem)
    print(f'Removed {rem} from FIELDS (no data in this period).')
print(f'Data available for: {list(data_detailed.keys())}')

### Verify download timestamps
Confirm the timestamps look right: they should be in **local time** (UTC+`TIMEZONE_OFFSET_TO_UTC_HOURS`) and mark the **end** of each averaging interval. The DB itself stores UTC — `InfluxIO` applied the offset on download. Eyeball the first/last stamps against the `START`/`STOP` you requested.

In [ ]:
for v in data_detailed.keys():
    idx = data_detailed[v].index
    print(f'{v}: index name={idx.name!r}, tz={idx.tz}, freq={idx.freqstr}')
    print(f'   first={idx[0]}   last={idx[-1]}')
print(f'\nApplied UTC offset: +{TIMEZONE_OFFSET_TO_UTC_HOURS}h (timestamps above are local time)')

### The raw record has two time resolutions
This variable is **10MIN until the March 2021 logger rebuild and 1MIN after it** (fieldbook 2021-03-24/26: the FF1 logger box was rebuilt and a new program uploaded; the sensors themselves were only reconnected, not replaced). The database keeps both eras under the same field name, so a single download returns a series whose spacing changes partway through.

`StepwiseMeteoScreeningDb` handles this itself, and it is worth knowing exactly how, because it shapes what the outlier tests see:

- it groups the records by detected resolution and **drops any group holding less than 0.2%** of the records (below, a 3-record 120-second group and ~200 ungrouped stragglers, all from the days the 0.3 m probe was failing in April 2024 and from the Dec 2025 logger uploads);
- it **upsamples the coarser era onto the finest grid**, back-filling each 10MIN value across the ten 1MIN slots it covers (the timestamp is `TIMESTAMP_END`, so the fill runs backwards);
- only then does it convert to `TIMESTAMP_MID` and start screening.

Two consequences for the tests further down: the 10MIN era arrives as **runs of ten identical values**, so any test built on *increments* sees nine zeros out of ten there and is not comparable across the two eras; and a window given in *records* covers the same wall-clock time in both eras, which is why windows below are written as `60 * 24 * ...`.

This is the single most consequential fact for the tests further down. The 10MIN era arrives as **runs of ten identical values**, so any test whose statistics are computed on *differences* sees nine zeros in every ten records there and degenerates: the rolling median absolute deviation of a mostly-constant series is `0`, so the detection band collapses. The practical rule is to check any test you add **per era**, not on its total count. The measurement for this depth is in *No spike test is used* below — where, unlike the pre-fix numbers reported for `0.05`, the current diive handles the degeneracy correctly and the differenced Hampel no longer mass-rejects the back-filled era. It still removes real soil moisture, which is why no spike test is committed.

> **Note the third era.** Beyond the 10MIN → 1MIN split (2021-03-26) this depth also carries the **sensor swap** at `SENSOR_SWAP` (2025-03-04). The two splits do **not** coincide, so per-era reporting below is explicit about *which* split it is showing: resolution for tests that care about resolution, sensor for tests that care about the sensor.

In [ ]:
_freqgroups = detect_freq_groups(index=data_detailed[FIELD].index)
_counts = _freqgroups.value_counts()
print(f'{"freq [s]":>10s} {"records":>10s} {"share":>8s}   first record        last record')
for _f in _counts.index:
    _ix = _freqgroups[_freqgroups == _f].index
    _share = len(_ix) / len(_freqgroups) * 100
    _kept = 'used' if _share > 0.2 else 'dropped by diive (< 0.2%)'
    print(f'{_f:10.0f} {len(_ix):10d} {_share:7.2f}%   {_ix[0]}  {_ix[-1]}   {_kept}')

### Plot downloaded high-res data

In [ ]:
for varname, frame in data_detailed.items():
    dv.plotting.TimeSeries(series=frame[varname]).plot()

## 🔍 Cross-check against the rest of the soil profile
Soil water content has no external reference — no weather service measures the soil at this plot. What it does have is **four other sensors in the same profile**, 5 to 50 cm below the same square metre of forest floor, plus the site's own **precipitation** record. Together those answer the question a single soil sensor cannot answer about itself: *is it still measuring the soil?*

The two failure modes worth separating:

- **it stops reporting** — communication failures, power loss. Loud and easy: gaps and impossible values, which the screening tests below remove.
- **it stops being coupled to the soil** — the soil shrinks away from the prongs in a dry summer, a root grows into the measurement volume, the backfill settles. This one is **quiet**: the series stays inside the physical range, stays smooth, and drifts plausibly. No statistical outlier test can see it. What gives it away is that the sensor no longer *responds*: rain falls, every deeper sensor wets up, and this one does not move.

> **What this can and cannot tell you.** The depths are not replicates — a 5 cm sensor legitimately dries faster, wets first, and swings harder than a 50 cm sensor, so the levels differ and the correlations are far from 1. What is *not* legitimate is a shallow sensor sitting still while the layers underneath it wet up: water reaches 20 cm by passing through 5 cm first. Use the section to *find candidate periods*, then confirm each one against the fieldbook before putting it in `REMOVE_DATES`.

*Sources: the FF1 soil profile (this site) and the screened precipitation product of notebook `30_PRODUCTS/08`.*

In [ ]:
def to_30min_mean(series: pd.Series, min_coverage: float = 0.5,
                  max_step: pd.Timedelta = pd.Timedelta('10min')) -> pd.Series:
    """Resample a high-res series to 30MIN means on the end-labelled grid.

    Uses the same end-labelled convention as the database: the period ending at T
    holds the records in (T-30min, T].

    A plain record count cannot decide whether a half hour is well covered here,
    because the raw spacing changes over the record (10MIN before the 2021 logger
    rebuild, 1MIN after) - three records mean full coverage in the first era and
    a 90% gap in the second. Each record is therefore credited with the time it
    represents, i.e. the distance back to the previous record, capped at
    *max_step* so the first record after a long gap cannot claim the whole gap.
    A half hour is returned only when the records present cover at least
    *min_coverage* of it, otherwise NaN - so a gap never masquerades as data.
    """
    if series.empty:
        raise ValueError('to_30min_mean got an empty series')
    spacing = series.index.to_series().diff()
    step = spacing.median()
    if pd.isna(step) or step > pd.Timedelta('30min'):
        raise ValueError(f'cannot resample to 30MIN from a {step} series')
    represented = spacing.clip(upper=max_step).fillna(step)
    grouper = pd.Grouper(freq='30min', label='right', closed='right')
    means = series.groupby(grouper).mean()
    coverage = represented.groupby(grouper).sum() / pd.Timedelta('30min')
    return means.where(coverage >= min_coverage)


def report_blocks(index, label: str, maxgap: str = '1D', note=None):
    """Group a DatetimeIndex of hits into contiguous blocks and print them
    ready to paste into REMOVE_DATES."""
    if len(index) == 0:
        print(f'{label}: none')
        return
    hits = pd.Series(index=pd.DatetimeIndex(index), data=1).sort_index()
    grp = (hits.index.to_series().diff() > pd.Timedelta(maxgap)).cumsum()
    print(f'{label}: {grp.nunique()} period(s)')
    for _, blk in hits.groupby(grp):
        comment = '' if note is None else f'  # {note(blk.index)}'
        print(f"    ['{blk.index[0]:%Y-%m-%d %H:%M:%S}', '{blk.index[-1]:%Y-%m-%d %H:%M:%S}'],{comment}")

#### Negative control
`to_30min_mean` would happily return a series of NaN if handed something that is not high-resolution — silently producing an "empty" comparison that looks like a data gap. Prove it raises instead.

In [ ]:
for _bad, _why in [(pd.Series(dtype=float, index=pd.DatetimeIndex([])), 'empty series'),
                   (pd.Series([1.0, 2.0, 3.0],
                              index=pd.date_range('2020-01-01', periods=3, freq='1h')),
                    'already coarser than 30MIN')]:
    try:
        to_30min_mean(_bad)
    except ValueError as e:
        print(f'PASSED - {_why} raises: {e}')
    else:
        raise AssertionError(f'(!) FAILED - to_30min_mean accepted {_why}')

### Download the companion depths
The other four depths, over the same period and from the same raw bucket. They are reduced to 30MIN means immediately and the high-resolution frames are dropped again — the cross-check needs half-hourly means, not four more million-record frames in memory.

⏳ This is the slowest cell in the notebook (four variables at raw resolution, several minutes).

In [ ]:
%%time
_comp_simple, _comp_detailed, _comp_measurements = dbc.download(
    bucket=BUCKET_RAW,
    measurements=[MEASUREMENT],
    fields=COMPANION_FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version=DATA_VERSION,
)

# Target first, then the companions ordered by depth: one 30MIN frame for the whole profile.
prof = pd.DataFrame({FIELD: to_30min_mean(data_detailed[FIELD][FIELD].dropna())})
for _v in COMPANION_FIELDS:
    if _v not in _comp_detailed:
        print(f'(!) {_v} has no data in this period and is left out of the cross-check.')
        continue
    prof[_v] = to_30min_mean(_comp_detailed[_v][_v].dropna())
prof = prof.sort_index()

# (!) Mask 0.05's decoupled window out of its companion column, so it does not
# distort THIS depth's cross-check (rolling agreement, rain response, profile mean).
# 0.05 has data there - it is just not coupled to the soil - so masking it in the
# companion frame is deliberate; it shows up as a 2020 coverage dip for 0.05 below.
for _c, (_a, _b) in COMPANION_MASK.items():
    if _c in prof.columns:
        prof.loc[_a:_b, _c] = np.nan
        print(f'masked {_c} {_a} -> {_b} out of the companion mean (its own documented fault).')

COMPANIONS = [c for c in prof.columns if c != FIELD]

del _comp_simple, _comp_detailed  # the high-res companion frames are not needed again

assert COMPANIONS, '(!) no companion depth available - the cross-check below cannot run'
print(f'profile frame: {prof.shape[0]} half-hours, {prof.index[0]} -> {prof.index[-1]}')
print(f'target: {FIELD} | companions: {COMPANIONS}')

### Level, coverage and correlation
First the plain description: how much data each sensor has, where it sits, and how strongly the depths move together. Correlations between depths are expected to be **high but not near-perfect** and to *decrease with depth separation* — the 5 cm and 50 cm sensors measure genuinely different things.

> ⚠️ **Every whole-record statistic here mixes two sensors.** The `describe()`, the correlation matrix and the mean below run across the 2025-03-04 swap, so the whole-record `mean` for `SWC_FF1_0.3_1` is an average of the old probe and the new one — not a property of either. The per-era split follows immediately, and the dedicated *The sensor swap* section quantifies the step. Read the whole-record numbers only as a coverage overview; read the level from the split.

(0.05 shows a 2020 coverage dip because its own decoupled window is masked out of the companion mean here — see *User settings*. That is deliberate and is not a fault of this depth.)

In [ ]:
print('data coverage per year (fraction of 30MIN periods with a value):')
display(prof.notna().groupby(prof.index.year).mean().round(3))

print('\nlevel and spread over the WHOLE record [% VWC] - (!) mixes both sensors:')
display(prof.describe().round(2))

# Split the target's own description at the swap - this is the honest view of level.
_old = prof[FIELD].loc[:OLD_PROBE_LAST]
_new = prof[FIELD].loc[SENSOR_SWAP:]
print('\n%s per era [%% VWC]:' % FIELD)
display(pd.DataFrame({'old probe (..2024-04-11)': _old.describe(),
                      'new probe (2025-03-04..)': _new.describe()}).round(2))

print('\ncorrelation of daily means, WHOLE record (mixes both sensors):')
display(prof.resample('D').mean().corr().round(3))

In [ ]:
_daily = prof.resample('D').mean()
fig, ax = plt.subplots(figsize=(14, 5))
for _c in prof.columns:
    _lw, _alpha = (2.0, 1.0) if _c == FIELD else (1.0, .55)
    ax.plot(_daily.index, _daily[_c], label=_c, lw=_lw, alpha=_alpha)
ax.set_ylabel('soil water content [%]')
ax.set_title(f'FF1 soil profile, daily means - {FIELD} highlighted')
ax.legend(ncol=len(prof.columns), fontsize=8)
plt.tight_layout()
plt.show()

## 🔀 The sensor swap
The one feature that makes this depth different from the other four: the field name `SWC_FF1_0.3_1` holds **two physical sensors**. The original 30 cm probe failed in April 2024 (and in failing took the whole FF1 SDI-12 bus down, which is why every depth has an April-2024 gap); it was replaced on 4 Mar 2025 by a probe set ~40 cm down the slope, in different soil.

This section establishes the eras **from the data**, verifies the dead period is genuinely empty, examines the run-up to the failure, and measures the level offset between the two probes. The boundaries are derived from the longest gap in the record itself; the constants `SENSOR_SWAP` and `OLD_PROBE_LAST` in *User settings* are then **asserted against** that result rather than trusted, and every era-split diagnostic in the notebook reads them. The offset is **reported, not applied** — both eras are uploaded as measured (see the top-of-notebook note and *Corrections*).

In [ ]:
# --- Establish the era boundaries FROM THE DATA ---
# Derived from the longest gap in the record, NOT from SENSOR_SWAP: the constant is
# verified against the data here, not trusted. (Slicing with .loc[:SENSOR_SWAP] would
# be wrong anyway - partial-string indexing includes the WHOLE swap day, so it would
# return a new-probe record as the old probe's last.)
_hires = data_detailed[FIELD][FIELD].dropna()
_spacing = _hires.index.to_series().diff()
_new_first = _spacing.idxmax()          # first record after the longest gap
_gap = _spacing.max()
_old_last = _new_first - _gap           # last record before it

print('ERA BOUNDARIES (derived from the longest gap in the record):')
print(f'  old probe  last record: {_old_last}  = {_hires.loc[_old_last]:.2f} % VWC')
print(f'  new probe first record: {_new_first}  = {_hires.loc[_new_first]:.2f} % VWC')
print(f'  dead period: {_gap}')

# The two constants in User settings must match what the data say.
assert pd.Timestamp(OLD_PROBE_LAST) == _old_last, (
    f'(!) OLD_PROBE_LAST={OLD_PROBE_LAST} but the data end at {_old_last}')
assert pd.Timestamp(SENSOR_SWAP).date() == _new_first.date(), (
    f'(!) SENSOR_SWAP={SENSOR_SWAP} but the new probe starts {_new_first}')
print('  PASSED - SENSOR_SWAP and OLD_PROBE_LAST agree with the data.')

# The fieldbook dates the unplugging to 2024-04-23; the data stop earlier.
_fieldbook_unplug = pd.Timestamp('2024-04-23')
print(f'\n  fieldbook unplug date: {_fieldbook_unplug.date()}  -> the probe went silent '
      f'{(_fieldbook_unplug - _old_last.normalize()).days} days EARLIER than the fieldbook says.')
print('  The era boundary is therefore dated from the data, not from the fieldbook.')

# --- Verify the dead period is genuinely empty (not present-but-constant) ---
_interior = _hires.loc[_old_last + pd.Timedelta('1min'):_new_first - pd.Timedelta('1min')]
print(f'\nDEAD PERIOD CHECK: {len(_interior)} records strictly inside the gap '
      f'(expected 0 -> genuinely absent, not stuck at a constant).')
assert len(_interior) == 0, '(!) records found inside the supposed dead period - investigate'

# The two station-wide gaps the other depths show in 2024 fall INSIDE this dead period.
for _lbl, _a, _b in [('2024-06-29 -> 07-01 (44 h)', '2024-06-29', '2024-07-01'),
                     ('2024-10-16 (8 h)', '2024-10-16', '2024-10-16')]:
    _inside = (_old_last < pd.Timestamp(_a)) and (pd.Timestamp(_b) < _new_first)
    print(f'  station gap {_lbl}: inside this depth\'s dead period = {_inside} '
          f'-> not visible in this record')

In [ ]:
# --- Examine the run-up to the failure: were the values degrading before silence? ---
_runup = prof.loc['2024-03-15':OLD_PROBE_LAST].resample('D').mean()
print('daily means, 2024-03-15 -> last record (target vs neighbours):')
display(_runup.round(2))
print('The 30 cm values track the profile in a normal spring dry-down right to the end - the\n'
      'probe stopped COMMUNICATING, it did not drift. No failing-but-reporting window to remove.')

fig, ax = plt.subplots(figsize=(14, 4))
for _c in prof.columns:
    _lw, _al = (2.0, 1.0) if _c == FIELD else (1.0, .5)
    ax.plot(prof.loc['2024-03-01':'2024-04-15'].index, prof.loc['2024-03-01':'2024-04-15', _c],
            label=_c, lw=_lw, alpha=_al)
ax.axvline(pd.Timestamp(OLD_PROBE_LAST), color='k', ls='--', lw=.8, label='last old-probe record')
ax.set_ylabel('soil water content [%]')
ax.set_title(f'{FIELD}: the run-up to the April 2024 failure')
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Measure the level offset between the two probes (REPORTED, not applied) ---
# A raw before/after mean is confounded by season (old era spans four years incl. dry
# summers; new era is only Mar-Dec 2025). Instead difference the target against a stable
# neighbour and match by calendar month, so like season is compared with like.
_daily_swap = prof.resample('D').mean()
_new_months = sorted(_daily_swap[FIELD].loc[SENSOR_SWAP:].dropna().index.month.unique())
print(f'new-probe era covers months {_new_months}\n')
print(f'{"reference":18s} {"old (matched months)":>22s} {"new":>10s} {"offset (new-old)":>18s}')
for _ref in ['SWC_FF1_0.2_1', 'SWC_FF1_0.5_1']:
    _d = _daily_swap[FIELD] - _daily_swap[_ref]
    _o = _d.loc[:OLD_PROBE_LAST]
    _o = _o[_o.index.month.isin(_new_months)]
    _n = _d.loc[SENSOR_SWAP:]
    print(f'{FIELD + " - " + _ref:18s} {_o.mean():22.2f} {_n.mean():10.2f} {_n.mean() - _o.mean():+18.2f}')
_dm = _daily_swap[FIELD] - _daily_swap[COMPANIONS].mean(axis=1)
_om = _dm.loc[:OLD_PROBE_LAST]; _om = _om[_om.index.month.isin(_new_months)]
_nm = _dm.loc[SENSOR_SWAP:]
print(f'{FIELD + " - profile":18s} {_om.mean():22.2f} {_nm.mean():10.2f} {_nm.mean() - _om.mean():+18.2f}')
print(f'\nraw era means (season-confounded, for contrast): '
      f'old {_daily_swap[FIELD].loc[:OLD_PROBE_LAST].mean():.2f} -> '
      f'new {_daily_swap[FIELD].loc[SENSOR_SWAP:].mean():.2f}')
print('=> new probe reads ~+3 % VWC higher relative to the profile. REPORTED, NOT CORRECTED.')

### Does the target still move with the profile?
Levels differ by depth, so compare **changes** rather than values: the daily change of the target against the daily change of the rest of the profile (with 0.05's decoupled window masked out), correlated in a rolling 30-day window. A well-coupled sensor keeps this high; a sensor that has lost contact with the soil keeps producing smooth, in-range values whose *changes* have nothing to do with the soil, and the rolling correlation collapses.

For this depth the rolling correlation dips several times, but every dip is the **known false-positive mode**: it happens when both the target and the profile are nearly flat (a dry summer, a frozen winter), so the correlation divides tiny changes by tiny changes and is dominated by noise. None of the dips is a loss of coupling — the rain-response table below is the test that settles each one, and the target responds through all of them. See *How to read the candidate list*.

In [ ]:
_d_target = _daily[FIELD].diff()
_d_others = _daily[COMPANIONS].mean(axis=1).diff()
rollcorr = _d_target.rolling(30, min_periods=20).corr(_d_others)

print('rolling 30-day correlation of daily increments, target vs profile mean:')
display(rollcorr.groupby(rollcorr.index.year).describe().round(2))

fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(rollcorr.index, rollcorr, color='tab:blue')
ax.axhline(0.3, color='tab:red', lw=.8, ls='--', label='0.3')
ax.set_ylim(-1, 1)
ax.set_ylabel('rolling r')
ax.set_title(f'{FIELD}: agreement of daily changes with the rest of the profile')
ax.legend()
plt.tight_layout()
plt.show()

report_blocks(rollcorr[rollcorr < 0.3].index, 'ROLLING AGREEMENT BELOW 0.3', maxgap='3D',
              note=lambda ix: f'{len(ix)} day(s)')

### Does the target still respond to rain?
For every day with more than 10 mm of precipitation, take the rise of each depth's daily mean over the following two days, and divide the target's rise by the profile mean's. A probe that has lost contact with the soil keeps producing smooth in-range values but stops *moving* when rain arrives, and this is the test that exposes it.

> **Read the ordering, not the absolute ratio — this is the second-deepest sensor.** A ratio near 0 is damning at 5 cm, where water arrives first; at 30 cm a summer event may genuinely not reach the sensor at all, so a low ratio in a dry year is unremarkable on its own. What must hold is the *ordering*: this depth should respond less than 0.05/0.1/0.2 and more than 0.5. It does, in both sensor eras. A collapse toward 0 **while the deeper 0.5 m sensor rises** would be the damning pattern — and it never happens here.

> 🔀 **Split at the swap.** The two eras are two different probes in different soil, so their response ratios are not one population. The new probe responds *more* sharply than the old one (it leads the profile where the old probe sat mid-pack), consistent with a fresh install in looser backfill. That is a sensor property, not a fault.

In [ ]:
prec = pd.read_parquet(PREC_PRODUCT)[PREC_COL]
prec.index = prec.index + pd.Timedelta('15min')  # TIMESTAMP_MID -> TIMESTAMP_END, as used here
_shared = prof.index.intersection(prec.index)
assert len(_shared) > 0.5 * len(prof), (
    '(!) precipitation product does not line up with the profile - check the timestamp shift')
print(f'precipitation: {prec.index[0]} -> {prec.index[-1]}, '
      f'{len(_shared) / len(prof) * 100:.1f}% of the profile period covered')

_dprec = prec.reindex(prof.index).resample('D').sum(min_count=40)
_events = _dprec[_dprec > 10].index

_rows = []
for _day in _events:
    _before = _daily.loc[:_day - pd.Timedelta('1D')].tail(1)
    _after = _daily.loc[_day:_day + pd.Timedelta('2D')]
    if _before.empty or _after.empty:
        continue
    _resp = _after.max() - _before.iloc[0]
    _resp['PREC'] = _dprec.loc[_day]
    _rows.append(_resp.rename(_day))
resp = pd.DataFrame(_rows)
resp['RATIO'] = resp[FIELD] / resp[COMPANIONS].mean(axis=1)
print(f'\n{len(resp)} rain events > 10 mm/day in the screened period')

print('\nmedian rise per depth [% VWC] after a > 10 mm day, per year:')
display(resp.groupby(resp.index.year).median().round(2))

print('\nmedian response ratio (target / profile mean), all events and summer only:')
_summer = resp[resp.index.month.isin([7, 8, 9, 10])]
display(pd.DataFrame({'all events': resp['RATIO'].groupby(resp.index.year).median(),
                      'Jul-Oct': _summer['RATIO'].groupby(_summer.index.year).median(),
                      'n events': resp['RATIO'].groupby(resp.index.year).size()}).round(2))

### Candidate periods for `REMOVE_DATES`
Four scans, each for a different failure mode. All of them produce **candidates only** — check each against the fieldbook and against the plots above before removing anything.

In [ ]:
_target_hires = data_detailed[FIELD][FIELD].dropna()

report_blocks(_target_hires[(_target_hires < SWC_MIN) | (_target_hires > SWC_MAX)].index,
              f'OUTSIDE THE PHYSICAL RANGE [{SWC_MIN}, {SWC_MAX}] %', maxgap='1D',
              note=lambda ix: f'{len(ix)} record(s)')

_cov = prof[FIELD].notna().resample('D').mean()
report_blocks(_cov[_cov < 0.9].index, 'DAILY COVERAGE BELOW 90%', maxgap='2D',
              note=lambda ix: f'{len(ix)} day(s)')

report_blocks(rollcorr[rollcorr < 0.3].index, 'NOT MOVING WITH THE PROFILE (rolling r < 0.3)',
              maxgap='3D', note=lambda ix: f'{len(ix)} day(s)')

# A stuck sensor repeats one value. Runs are measured in *time*, not in records,
# because a record means ten minutes in the early era and one minute in the late one.
# Stored precision also changed: until the logger software update of 21 Apr 2020 the
# values arrived in steps of 0.1%, so during a slow dry-down they legitimately hold the
# same rounded value for hours. The threshold below is therefore loose, and the maxima
# printed with it - split at the resolution boundary AND the sensor swap - are the more
# informative numbers.
_runs = (_target_hires != _target_hires.shift()).cumsum()
_spans = _target_hires.groupby(_runs).apply(lambda x: x.index[-1] - x.index[0])
_ends = _target_hires.groupby(_runs).apply(lambda x: x.index[-1])
_sizes = _target_hires.groupby(_runs).size()
_stuck = _spans[_spans >= pd.Timedelta('24h')]
print(f'IDENTICAL-VALUE RUNS LASTING >= 24 h: {len(_stuck)}')
for _g in _stuck.index:
    print(f"    ['{_ends.loc[_g] - _spans.loc[_g]:%Y-%m-%d %H:%M:%S}', "
          f"'{_ends.loc[_g]:%Y-%m-%d %H:%M:%S}'],  # {_sizes.loc[_g]} identical records")
for _label, _mask in [('whole record', _spans.notna()),
                      ('10MIN era (to 2021-03-26)', _ends <= pd.Timestamp('2021-03-27')),
                      ('1MIN era, old probe', (_ends > pd.Timestamp('2021-03-27'))
                       & (_ends < pd.Timestamp(SENSOR_SWAP))),
                      ('new probe (from 2025-03-04)', _ends >= pd.Timestamp(SENSOR_SWAP))]:
    _sub = _spans[_mask]
    if len(_sub) == 0:
        print(f'   longest identical-value run, {_label}: none')
        continue
    print(f'   longest identical-value run, {_label}: {_sub.max()} '
          f'(ending {_ends.loc[_sub.idxmax()]})')

print('\n(Copy any confirmed period into REMOVE_DATES below. Check the fieldbook first.)')

### Adjudicate the candidates against the fieldbook
A candidate period means nothing until it is matched against what happened at the site. The GIN
fieldbook export is read here directly, so the adjudication written up below can be re-run and
challenged rather than taken on trust.

Filtering on the SWC device tags alone would miss most of it: the entries that explain these periods
are power-supply and logger entries carrying no soil device at all. The filter therefore also takes
anything at this profile's location, its logger/DAQ, or a free-text mention of soil, power, logger
or SDI-12.

In [ ]:
import html
import re

_fb = pd.read_csv(FIELDBOOK, encoding='utf-8-sig')
_fb.columns = [c.strip() for c in _fb.columns]
_fb['date'] = pd.to_datetime(_fb['Date'], format='%d.%m.%Y', errors='coerce')
assert _fb['date'].notna().all(), '(!) unparsed dates in the fieldbook - check the date format'


def _plaintext(s):
    """GIN stores the entry body as HTML; strip the tags and entities."""
    return re.sub(r'\s+', ' ',
                  html.unescape(re.sub(r'<[^>]+>', ' ', str(s))).replace('\xa0', ' ')).strip()


_fb['text'] = _fb['Text'].map(_plaintext)
_mask = (_fb['Operation Tag'].astype(str).str.contains(r'SWC|iDL_FF1|iDAQ_FF1', case=False, na=False)
         | _fb['Location'].astype(str).str.contains(PROFILE, case=False, na=False)
         | _fb['text'].str.contains(r'soil|SWC|Teros|SDI|power|logger', case=False, na=False))
fb = (_fb[_mask & _fb['date'].between(START, STOP)]
      .drop_duplicates(subset=['date', 'Event Tag', 'Text'])
      .sort_values('date'))
assert len(fb) > 0, '(!) no fieldbook entries matched - has the export format changed?'
print(f'fieldbook: {len(_fb)} entries total, '
      f'{len(fb)} relevant to {PROFILE} within the screened period')


def fieldbook_near(start, stop=None, days: int = 3, maxchars: int = 220):
    """Print the fieldbook entries within *days* of a period."""
    start = pd.Timestamp(start)
    stop = pd.Timestamp(stop) if stop is not None else start
    hits = fb[fb['date'].between(start - pd.Timedelta(days=days), stop + pd.Timedelta(days=days))]
    if hits.empty:
        print(f'   (no fieldbook entry within +/-{days} days)')
        return
    for _, r in hits.iterrows():
        print(f"   {r['date']:%Y-%m-%d} {r['Event Tag']:<12s} {r['text'][:maxchars]}")

In [ ]:
# One entry per candidate period found by the scans above - REBUILT from THIS depth's
# own scan output (not carried over from 0.05). It is the record of what was adjudicated.
CANDIDATES = [
    ('2021-03-24', '2021-04-02', 'coverage gap (logger rebuild)'),
    ('2022-09-06', '2022-09-16', 'coverage gap (FF1 power supply)'),
    ('2023-06-15', '2023-06-15', 'coverage gap (FF1 power incident)'),
    ('2024-04-09', '2025-03-04', 'the dead period: old probe silent -> new probe installed'),
    ('2025-12-18', '2025-12-19', 'out-of-range values (logger-program uploads)'),
    # rolling-agreement dips, all near-zero-change false positives (see prose):
    ('2021-10-22', '2021-12-30', 'rolling r < 0.3 (flat autumn/winter, kept)'),
    ('2022-07-05', '2022-08-04', 'rolling r < 0.3 (flat dry summer, kept)'),
    ('2023-08-13', '2023-11-15', 'rolling r < 0.3 (flat/low-change, kept)'),
]
for _start, _stop, _why in CANDIDATES:
    print(f'\n--- {_start} -> {_stop}  ({_why})')
    fieldbook_near(_start, _stop, days=3)

#### How to read the candidate list

The outcome for `SWC_FF1_0.3_1`, read off the cell above. The organising fact is that this depth is **two sensors**: the old probe (to 2024-04-11), the dead period, and the new probe (from 2025-03-04). No candidate below is a *value* fault in either working era — the only removal is the handful of Dec 2025 sentinels, and the dead period is simply absent.

- **Out of physical range** — **7 records**, all in a single cluster on **18-19 Dec 2025**, all fixed negatives (−69.56, −19.32). They are the signature of a failed SDI-12 read (the CR1000 firmware stores a failed read as a fixed negative number, not a NaN) and coincide exactly with two remote logger-program uploads to the FF1 meteo logger on those days. They are removed **without hand-adjudication**, by two mechanisms: **5 of the 7** sit in the irregular-frequency stragglers of those uploads, so diive discards them with the sub-0.2% frequency groups before screening; the **other 2** reach the **absolute-limits test**. Either way the rule is physical. Each is an isolated minute in an otherwise full half hour, so removing them costs no half-hourly value. **Note the asymmetry with the other depths:** this probe produced **no** negative sentinels during the April 2024 bus failure at all — it simply stopped returning data. Its four neighbours each logged 12-21 sentinels that April (0.05: 20, 0.1: 21, 0.2: 12, 0.5: 14), because the *failing 0.3 probe took their shared SDI-12 bus down* while it itself went silent. So absolute limits does *less* work at this depth than at any other (7 vs 19-28), and all 7 of ours are the unrelated Dec 2025 event.
- **Coverage below 90%** — **four** periods, and this is where the dead period shows up. Three are the profile-wide events every depth shares (the 2021-03 logger rebuild, the 2022-09 FF1 power-supply failure, the 2023-06-15 FF1 power incident). The fourth is a **single 330-day block, 2024-04-09 → 2025-03-04** — the dead period. Because the two unexplained station gaps (2024-06-29 → 07-01 and 2024-10-16) and the April-2024 bus interruption all fall *inside* this depth's dead period, they do **not** appear here as separate blocks the way they do at the other four depths. That is correct — there is nothing to see because the sensor was gone. Gaps need no removal; they are already missing.
- **Not moving with the profile** — several dips of the rolling correlation, **all false positives**. Every one falls in a stretch where both the target and the profile are nearly flat (autumn/winter 2021, the dry summer of 2022, low-change spells of 2023), so the rolling correlation divides tiny day-to-day changes by tiny changes and collapses on noise. Checked event by event, the sensor **responds** through all of them: e.g. 4 Dec 2021 (+5.5% on 11.8 mm), 12-14 Nov 2023 (ratios 0.6 and 1.1). None is a loss of coupling, and there is **no decoupling fault at this depth** — unlike 0.05's real 2020 fault, the rain-response ratios here never collapse toward 0 while the profile rises. **All kept.**
- **Identical-value runs** — **two runs over 24 h**, and both fall in the **11 days 2020-04-10 → 2020-04-21**, when values arrived in steps of exactly 0.1% and this depth was drying down slowly enough in spring to hold a rounded value for more than a day (28.7% held 25 h). That ends with the logger software change of 21 Apr 2020 (two-byte precision). A storage-resolution artefact, not a stuck sensor. In both 1MIN eras no value ever repeats for more than **two minutes**. (0.05 saw no 24-h run because it swings faster; the slower 30 cm sensor holding a rounded value longer is expected, not a fault.)

So nothing here needs a manual removal window: the physical rule handles Dec 2025, the dead period is already missing, and every rolling-agreement dip is a false positive. `REMOVE_DATES` is **empty** for this depth.

## ▶️ Start MeteoScreening with `diive`

In [ ]:
mscr = dv.qaqc.StepwiseMeteoScreeningDb(
    site=SITE,
    data_detailed=data_detailed,
    fields=FIELDS,
    site_lat=SITE_LAT,
    site_lon=SITE_LON,
    utc_offset=TIMEZONE_OFFSET_TO_UTC_HOURS,
)
mscr.showplot_orig()

## 🔍 Outlier detection
Run a test → inspect its preview → commit with `mscr.addflag()`. Only the committed flag of the **most recent** test is added. Skip any test a variable does not need.

**What soil water content needs, and what it does not.** The series is smooth, strongly autocorrelated, and seasonally non-stationary: it ranges over many percentage points between a wet winter and a dry August, and the wet and dry extremes are the most interesting values in it, not the least trustworthy. Two consequences run through this whole section:

- **Never split day and night.** Every test below is run with `separate_daytime_nighttime=False`. Soil moisture has no meaningful diurnal cycle at these depths.
- **Distribution-wide tests do not belong here.** A z-score over the whole record measures how far a value sits from the multi-year mean, which for this variable is a statement about the season — and here it is *also* corrupted by the sensor swap, since the multi-year mean straddles two probes. Those tests are listed further down, switched off, with the reasoning.

**For this depth the list is especially short.** There is no documented value fault in either working era and `REMOVE_DATES` is empty, so screening is just the failed-read sentinels (**absolute limits**) and the **missing-values** flag. The dead period is handled by the missing-values flag like any gap. No spike test is committed — see *No spike test is used* below.

In [ ]:
mscr.start_outlier_detection()

Plot the current cleaned data at any point during detection:

In [ ]:
for key, val in mscr.outlier_detection.items():
    val.showplot_cleaned(interactive=False)

### Manual removal
Flag specific timestamps or time ranges for removal — known sensor failures, maintenance windows, etc. Give `[start, stop]` pairs and/or single timestamps.

**This depth has none.** Manual removal exists for values that are *wrong but plausible* — the one class no physical rule can catch (0.05's decoupled probe is the example). At 30 cm every candidate the scans threw up is either handled by a rule (the Dec 2025 sentinels → absolute limits), already missing (the 327-day dead period), or a false positive (the rolling-agreement dips). So `REMOVE_DATES` is empty and both cells below skip themselves. A manual window is a blunt instrument — it deletes a whole range, not the offending minutes — so it is not used where a rule will do.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.ManualRemoval)

In [ ]:
# Windows for THIS depth (30 cm). Re-derived from this sensor's own evidence; the
# 5 cm window from the template has been DELETED (it belongs to that probe alone).
#
# There is nothing to put here. The candidate scan and its adjudication (above) found:
#   - the only out-of-range values (Dec 2025) are handled by absolute limits;
#   - the dead period 2024-04-11 -> 2025-03-04 is already missing;
#   - every rolling-agreement dip is a near-zero-change false positive, and the sensor
#     responds to rain through all of them (no decoupling fault at this depth);
#   - the run-up to the failure is clean: the 30 cm values track the profile normally
#     right up to the last record on 2024-04-11 (see 'The sensor swap'). The probe
#     went from reporting-normally to silent - a communication failure, not a stretch
#     of wrong-but-plausible values - so there is no failing-but-reporting window to cut.
# Manual removal is reserved for values that are WRONG BUT PLAUSIBLE, the one class no
# rule can catch. This depth has none, so the list is empty and the two guarded
# manual-removal cells skip themselves.
REMOVE_DATES = []
if REMOVE_DATES:
    mscr.flag_manualremoval_test(remove_dates=REMOVE_DATES, showplot=True, verbose=True)
else:
    print('No manual removal windows for this depth - skip the addflag cell below.')

In [ ]:
if REMOVE_DATES:
    mscr.addflag()

### Absolute limits
Flags values outside a fixed physical range `[minval, maxval]`. This is the workhorse test for soil water content across the profile: a failed SDI-12 read is stored as a fixed negative number rather than a NaN, and every one of them lands far outside the physical range.

**At this depth it does less work than at any other.** The old 30 cm probe produced **no** sentinel reads at all — when it failed it simply stopped returning data, while taking the shared SDI-12 bus down with it (which is why its four neighbours each logged 12-21 sentinels in April 2024 - 0.05: 20, 0.1: 21, 0.2: 12, 0.5: 14 - and this one logged none). The only out-of-range records here are the **7** of 18-19 Dec 2025, from two remote logger-program uploads - an event that hit all five depths equally, exactly 7 each, and has nothing to do with this probe. **5 of those are already gone** with the sub-0.2% frequency stragglers — so this test removes **2 records**, both in the new-probe era.

In [ ]:
if SHOW_PARAM_HELP:
    help(dv.outliers.AbsoluteLimits)

In [ ]:
# Volumetric water content in %, so 0 is the hard physical floor. The upper limit is
# deliberately loose: the highest value ever recorded at this depth is 36.5% (either
# era), well below any plausible porosity, so 60 can only fire on something unphysical.
# A failed SDI-12 read enters the record as that sensor's calibration polynomial
# evaluated at zero response, a fixed negative value (-19.32, -69.56) - hence a floor
# of 0 catches all of them and nothing else. At THIS depth the test does little work:
# the only out-of-range records are the 7 of 18-19 Dec 2025, of which 5 are already
# gone with the stray-frequency stragglers, so absolute limits removes just 2 records
# (both new-probe era). The old probe produced NO sentinels - it went silent instead.
mscr.flag_outliers_abslim_test(minval=SWC_MIN, maxval=SWC_MAX, showplot=True, verbose=True)

In [ ]:
mscr.addflag()

### No spike test is used, and the differenced Hampel filter still must not be committed
Every remaining template test (Hampel, z-score, rolling z-score, local SD, increments z-score, local outlier factor, trim low) is **left switched off**. Measured on *this* record with the current (fixed) diive, the differenced Hampel no longer mass-rejects the back-filled era — but it still removes real soil moisture, so it stays off. The numbers, run after absolute limits and split by era:

| Hampel call | 10MIN era (back-filled) | 1MIN era, old probe | new probe |
|---|---|---|---|
| `use_differencing=True, n_sigma=8` | 24 (0.00%) | 841 (0.04%) | 40 (0.01%) |
| `use_differencing=True, n_sigma=30` | 8 | 0 | 9 |
| `use_differencing=True, n_sigma=100` | 0 | 0 | 6 |
| `use_differencing=False, n_sigma=8` | 11 876 (2.35%) | 10 911 (0.53%) | 2 114 (0.49%) |

Two things to read here. First, the **fix works**: where the *pre-fix* diive removed 19.4% of the 10MIN era at 0.05 (the rolling MAD of a mostly-constant back-filled series is `0`, and substituting a tiny epsilon collapsed the band), the current version leaves those records alone — 24 out of 504 550, and `n_sigma` finally does something (24 → 8 → 0). The structural trap is still real and still worth checking per era; diive just no longer falls into it. Second, and decisively, **the differenced Hampel still flags real wetting fronts**: at `n_sigma=8` its ~900 hits include the two largest, genuinely rain-driven jumps in the record. The non-differenced Hampel is worse, removing 2.35% of the back-filled era outright.

**There is nothing for a spike test to find.** Once absolute limits has removed the 2 reachable failed reads, the largest remaining minute-to-minute excursions are real:

- **2025-07-27 16:14** (new probe): 25.3 → 34.8% over three minutes, with **21.3 mm** of rain in the 16:30 half hour — and still at 30.7% six hours later.
- **2021-06-28 18:21** (old probe): 26.4 → 31.1% over ~8 minutes, with **11.6 + 8.2 mm** of rain in the window — and still at 30.9% four hours later.

Those are wetting fronts, the most physically real feature in the series. The 99.99th percentile of minute-to-minute change is 0.19%; the only larger jumps are these fronts and the edges of the dead period. Any filter tuned tightly enough to fire would remove the fronts.

The others, for the record: **z-score over the whole record** measures distance from the multi-year mean, which for this variable *is* the season — and here it is doubly wrong, because the multi-year mean averages two different sensors across the swap. **Rolling z-score / local SD** fight the same physics as Hampel. **Increments z-score** degenerates in the back-filled era exactly as the differenced Hampel used to. **Local outlier factor** is far too slow on 1MIN data. **Trim low** removes a fixed share of low values by construction.

So screening here is: **absolute limits** (physical range) + **missing values**. No manual removal (nothing wrong-but-plausible at this depth), and no spike test.

One practical warning for any test you do try: differencing-based tests compute their differences **after dropping missing records**, so the records flanking every gap are compared across the gap and look like spikes. This record has the **327-day dead period** (2024-04-11 → 2025-03-04) and a 10-day gap in September 2022 — the fixed diive excludes cross-gap differences, but check the preview anyway.

If you want to try one, the calls are below. Run it, look at the preview, check what it does to the 2020-2021 back-filled era **and** to the two wetting fronts specifically, and only `mscr.addflag()` if it is genuinely removing artefacts.

In [ ]:
# Optional, all off by default - read the note above before enabling any of these,
# and check the flagged count per era rather than the total.
# mscr.flag_outliers_hampel_test(window_length=60 * 24, n_sigma=8, use_differencing=False,
#                                separate_daytime_nighttime=False,
#                                repeat=False, showplot=True, verbose=3)
# mscr.flag_outliers_zscore_test(thres_zscore=4.5, separate_daytime_nighttime=False,
#                                repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_zscore_rolling_test(thres_zscore=4.5, winsize=60 * 24 * 7,
#                                        repeat=True, showplot=True, verbose=True)
# mscr.flag_outliers_localsd_test(n_sd=7, winsize=60 * 24 * 7, constant_sd=False,
#                                 separate_daytime_nighttime=False,
#                                 repeat=False, showplot=True, verbose=True)
# mscr.flag_outliers_increments_zcore_test(thres_zscore=40, repeat=True, showplot=True, verbose=True)
# mscr.addflag()

### Missing values
Not an outlier test — flags missing records so they are counted in the overall `QCF`.

In [ ]:
mscr.flag_missingvals_test(verbose=True)

### Overall quality flag QCF
Aggregate all committed test flags into one overall flag `QCF` (0 = good, 1 = marginal, 2 = bad) and filter the series. Required before corrections and resampling.

In [ ]:
mscr.finalize_outlier_detection()

#### Reports

In [ ]:
mscr.report_outlier_detection_qcf_evolution()

In [ ]:
mscr.report_outlier_detection_qcf_flags()

In [ ]:
mscr.report_outlier_detection_qcf_series()

#### Plots

In [ ]:
mscr.showplot_outlier_detection_qcf_heatmaps()
# mscr.showplot_outlier_detection_qcf_timeseries()

## 🔧 Corrections
Applied to the high-res, QCF-filtered data. **None of them apply to soil water content**, and all calls below stay commented out.

The corrections diive offers are offset removals for radiation and relative humidity, threshold clipping, and setting ranges to a constant — every one of them writes a value the sensor did not measure. For this variable there is nothing to correct *within* the series: a probe reading too high or too low is a calibration matter (soil-specific permittivity calibration), which cannot be fixed after the fact from the series itself.

> 🔀 **In particular, the sensor swap is deliberately NOT corrected here.** The new probe reads about +3 % VWC higher relative to the profile than the old one (see *The sensor swap*), but that step is left in: it is a real difference between two sensors in two soil locations, the +3 % is a season-confounded estimate, and applying it would fabricate values the new probe never measured. The break is documented, not homogenised — both eras go to the database as measured.

> ⚠️ Also do **not** clip to a minimum: the failed reads are already gone via absolute limits, and clipping would replace a missing measurement with a fabricated 0% — which reads as "bone dry soil" downstream.

In [ ]:
mscr.showplot_cleaned()

Inspect the most frequent values first (read-only, safe to run). For soil water content there should be **no** dominant repeated value: a value appearing thousands of times in the 1MIN era would mean a stuck sensor. Frequent repeats *are* expected in the 2020-2021 part, where each 10MIN reading was back-filled across ten 1MIN slots.

In [ ]:
for ff in mscr.fields:
    vc = mscr.series_hires_cleaned[ff].value_counts()
    print(f'--- {ff} (top 20 of {mscr.series_hires_cleaned[ff].count()} records) ---')
    print(vc.head(20))

In [ ]:
# All commented out on purpose - none of these apply to soil water content.
# mscr.correction_setto_max_threshold(threshold=60)
# mscr.correction_setto_min_threshold(threshold=0)
# mscr.correction_setto_value(dates=[['2022-04-01', '2022-04-05']], value=0, verbose=1)
# mscr.correction_set_exact_value_to_missing(values=[0])
# mscr.showplot_cleaned(interactive=False)

## 🔁 Resampling

### Resample to 30MIN
Resample the screened high-res series to `RESAMPLING_FREQ`. The output timestamp is `TIMESTAMP_END` again (see *Timestamp convention*), ready for upload.

In [ ]:
# mincounts_perc stays at the template default of .25, the opposite choice from the
# precipitation notebook. A 30MIN *mean* of a smooth state variable is well estimated
# from a quarter of the records, whereas a *sum* built from a quarter of the records
# under-reports by construction. Requiring near-complete coverage here would throw
# away good half-hours during the April 2024 bus trouble for no gain in accuracy.
mscr.resample(to_freqstr=RESAMPLING_FREQ, agg=RESAMPLING_AGG, mincounts_perc=.25)
mscr.showplot_resampled()

### Check the resampled time resolution

In [ ]:
for v in mscr.resampled_detailed.keys():
    freq = dv.times.DetectFrequency(index=mscr.resampled_detailed[v].index, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: {freq}')

### Re-check the screened product against the profile
The same comparison as before screening, now on the resampled 30MIN product. Screening should leave the agreement with the rest of the profile **no worse** than it was — if the correlation drops, a test removed real soil moisture rather than an artefact. For this depth almost nothing is removed (2 sentinel minutes), so the agreement should be essentially unchanged.

> The audit is run **per era** (old probe vs new probe). A single correlation across the swap would be meaningless, because the increments either side belong to different sensors in different soil.

In [ ]:
_screened = mscr.resampled_detailed[FIELD][FIELD]

_before = prof.resample('D').mean()
_after = _before.copy()
_after[FIELD] = _screened.resample('D').mean()
_compmean_b = _before[COMPANIONS].mean(axis=1)
_compmean_a = _after[COMPANIONS].mean(axis=1)

print(f'{"era":26s} {"n 30MIN":>9s} {"r_before":>9s} {"r_after":>9s} '
      f'{"mean [%]":>9s} {"removed":>8s}')
for _lbl, _a, _b in [('old probe (..2024-04-11)', None, OLD_PROBE_LAST),
                     ('new probe (2025-03-04..)', SENSOR_SWAP, None)]:
    _rb = _before[FIELD].loc[_a:_b].diff().corr(_compmean_b.loc[_a:_b].diff())
    _ra = _after[FIELD].loc[_a:_b].diff().corr(_compmean_a.loc[_a:_b].diff())
    _nb = prof[FIELD].loc[_a:_b].notna().sum()
    _na = _screened.loc[_a:_b].notna().sum()
    _mn = _screened.loc[_a:_b].mean()
    print(f'{_lbl:26s} {_na:9d} {_rb:9.3f} {_ra:9.3f} {_mn:9.2f} {_nb - _na:8d}')
    if _ra < _rb - 0.01:
        print(f'   (!) WARNING - {_lbl}: agreement with the profile got worse.')
n_before = prof[FIELD].notna().sum()
n_after = _screened.notna().sum()
print(f'\ntotal removed: {n_before - n_after} of {n_before} half-hours that had a value '
      f'({(1 - n_after / n_before) * 100:.3f}%)')
print('OK - the two removed records are the reachable Dec 2025 sentinels; neither empties a '
      'half-hour, and agreement is unchanged in both eras.')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(prof.index, prof[FIELD], label=f'{FIELD}, unscreened', color='tab:blue', alpha=.4, lw=.8)
ax.plot(_screened.index, _screened, label=f'{FIELD}, screened', color='tab:blue', lw=.8)
ax.plot(prof.index, prof[COMPANIONS].mean(axis=1), label='profile mean (other depths)',
        color='tab:orange', lw=.8)
ax.set_ylabel('soil water content [%]')
ax.set_title('Screening effect against the rest of the profile')
ax.legend()
plt.tight_layout()
plt.show()

## ⬆️ Upload data to database

**Re-uploading overwrites the same variant — safe to re-run.** With `delete_from_db_before_upload=True` (below), the upload first *deletes*, then writes. The delete is scoped to the exact match `_measurement` + `varname` + `data_version` (`meteoscreening_diive`) over the uploaded time range, so re-screening a period replaces only its previous screened result. It never touches the raw data (different `data_version`, and a different `_raw` bucket), other variables, or other data versions. The delete-first step (rather than a plain overwrite) matters because InfluxDB keys a point by its full tag set: if a tag changed between runs (e.g. `units`, `gain`, `offset`), a plain write would leave the old point as a **duplicate** — the delete removes it regardless of tags.

Only the target variable is uploaded. The companion depths were downloaded read-only for the cross-check and each has its own notebook.

In [ ]:
print(f'Uploading to bucket {BUCKET_PROCESSED}')
for v in mscr.resampled_detailed.keys():
    dbc.upload_singlevar(
        to_bucket=BUCKET_PROCESSED,
        to_measurement=assigned_measurements[v],
        var_df=mscr.resampled_detailed[v],
        timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
        delete_from_db_before_upload=True,
    )

### Verify upload
Download the just-uploaded data back and confirm the time resolution and timestamps. The offset is the same `TIMEZONE_OFFSET_TO_UTC_HOURS`, so the timestamps below should again be local `TIMESTAMP_END` — matching what you screened.

In [ ]:
# Fresh variable names so the screened originals (data_detailed etc.) are not overwritten:
check_simple, check_detailed, check_measurements = dbc.download(
    bucket=BUCKET_PROCESSED,
    measurements=[MEASUREMENT],
    fields=FIELDS,
    start=START,
    stop=STOP,
    timezone_offset_to_utc_hours=TIMEZONE_OFFSET_TO_UTC_HOURS,
    data_version='meteoscreening_diive',
)
check_simple

In [ ]:
for v in check_detailed.keys():
    idx = check_detailed[v].index
    freq = dv.times.DetectFrequency(index=idx, verbose=True).get()
    status = 'PASSED' if freq == RESAMPLING_FREQ else '(!) FAILED'
    print(f'{status} - {v}: freq={freq}, first={idx[0]}, last={idx[-1]}')

## ✅ End of notebook

In [ ]:
_end = datetime.now()
print(f"Finished: {_end.strftime('%Y-%m-%d %H:%M:%S')}  "
      f"(runtime {str(_end - NOTEBOOK_START).split('.')[0]})")

***
### 📝 Notes for this variable
- **Mean, not sum.** `RESAMPLING_AGG = 'mean'` — soil water content is a state, not a flux.
- **Units are volumetric %,** stored with `gain = 100` on a raw `m^3/m^3` reading. The database unit tag says `%` for both eras; the magnitudes (20-36) agree with that.
- **🔀 Two sensors under one field name.** The 30 cm probe installed 19 Mar 2020 failed in April 2024 and was replaced on **4 Mar 2025** by a probe set **40 cm down the slope**, in different soil. The record is therefore an *old-probe era* (2020-04-10 → 2024-04-11), a **327-day dead period**, and a *new-probe era* (2025-03-04 → present). The two eras are uploaded **as measured, with no level adjustment**; the series is **not homogeneous across 2025-03-04**. See *The sensor swap*.
- **Sensor family.** Both probes are **TEROS 12**; the logger box was rebuilt 24-26 Mar 2021, which changed the resolution from 10MIN to 1MIN but not the sensors. **Nothing before 2020-04-10 belongs to this series** — soil moisture up to that day lives under the old `SWC_M5_*` names (a different profile, different sensors, gain 1).
- **Three eras, three boundaries, and they do not coincide.** Resolution splits at 2021-03-26 (10MIN → 1MIN); the sensor splits at 2024-04-11 (silent) and 2025-03-04 (replaced). Report each test's count on the split it cares about — resolution for differencing tests, sensor for level.
- **The first 11 days are coarse.** From 2020-04-10 to 2020-04-21 values arrive in steps of 0.1% VWC; a logger software change on 21 Apr 2020 moved them to full float precision. At this depth the slow spring dry-down held a rounded value for more than a day twice in that window — a storage artefact, not a stuck sensor.
- **A failed sensor read is a number, not a gap.** The CR1000 firmware predates SDI-12 NaN support, so a failed read lands as a fixed negative value (−19.32, −69.56 at this depth). Absolute limits, not the missing-values flag, catches them — but at this depth there are only 7, all on 18-19 Dec 2025, and only 2 reach the test (the other 5 go with the stray-frequency stragglers).
- **The profile is the reference, and it is not a replicate set.** Nothing from the companion depths is written into this series or uploaded. 0.05's decoupled 2020 window is masked out of the companion mean here — that is 0.05's fault, not this depth's.
- **Exports a 327-day gap.** Downstream users get NaN across the dead period (2024-04-11 → 2025-03-04); it is not gap-filled here.

***
### 🔀 The sensor swap: the central issue of this depth

This is the dominant feature of `SWC_FF1_0.3_1`, and the reason this notebook carries a section the other four depths do not. What the **data** show (established in *The sensor swap* diagnostics, not just taken from the fieldbook):

- **Old probe, last record: `2024-04-11 01:46`** (value 27.13%). **New probe, first record: `2025-03-04 14:36`** (value 28.76%). The gap between them is **327 days 12:50** and is genuinely empty — **zero** records fall inside it, so nothing was reported present-but-constant.
- **The probe went silent ~12 days before it was unplugged.** The fieldbook dates the unplugging to **23 Apr 2024**; the data stop on **11 Apr 2024**. The era boundary is therefore taken from the data (`OLD_PROBE_LAST`), not the fieldbook. The failure was in the SDI-12 communication — which is exactly what "took the whole FF1 bus down", producing the April-2024 coverage gap seen at *every* depth.
- **The run-up to the failure is clean.** Through late March and early April 2024 the 30 cm daily means track the whole profile in a normal spring dry-down, in range, with no flat runs and no loss of rain response, right up to the last record. The probe did not degrade its *values* before dying; it stopped *communicating*. There is therefore **no failing-but-reporting stretch to remove** — `REMOVE_DATES` stays empty. (The last few April-9 records arrive at irregular spacing as the bus began to fail; diive drops those with the sub-0.2% frequency groups, so they never reach screening.)
- **The level offset is about +3 % VWC, and it is NOT corrected.** Measured as the month-matched difference of daily means against the nearest stable neighbours (0.2 m, r ≈ 0.95 in both eras; and the masked profile mean), the new probe sits **+3 to +4.5 % VWC higher relative to the profile** than the old one — e.g. against 0.2 m the spring (Mar-Jun) gap moves from about −5.2 to about −0.5, a step of ≈ +4.7; the full-year month-matched figures are +3.3 (vs 0.2), +2.7 (vs 0.5) and +4.3 (vs the profile mean). The estimate is confounded by season (the old era spans four full years including dry summers; the new era is only Mar-Dec 2025) and by the genuinely different soil location 40 cm downslope, which is exactly why it is reported as evidence and **not** applied. A raw before/after mean (26.7 → 29.3, +2.5) understates it and is season-confounded; the neighbour-differenced, month-matched estimate is the honest one.
- **The new probe responds more sharply to rain than the old one** (its median post-event rise leads the profile, where the old probe sat mid-pack), consistent with a fresh install in disturbed, looser backfill. That is a property of the new sensor, not a fault.

**Treatment (settled).** Screen both eras under the one field name; upload both **as measured**, with **no** offset correction, no dropping of the post-swap era, and no split into a second field name. Document the break — this section and the top-of-notebook note. Rationale: an offset correction would write values the sensor did not measure; the ~3 % is confounded by season and by a different soil location; a documented break is honest where a silent homogenisation is not. The swap date lives in `SENSOR_SWAP` so the decision is easy to find and change.

### ✅ There is no decoupling fault at this depth

Unlike 0.05 (whose 2020 fault is why this whole cross-check exists), the 30 cm sensor stays coupled to the soil in both eras. Every rolling-agreement dip is a near-zero-change false positive, and the rain-response ratio never collapses toward 0 while the profile rises. The rain-response ordering is physically sensible in both eras (this depth sits mid-profile, weaker than the shallow sensors, stronger than 0.5 m). So the only screening actions are absolute limits (2 records) and the missing-values flag.

### 🔎 Fieldbook cross-check (GIN export `CH-LAE-laegeren-export_20260719.csv`)

The export is read in the *Adjudicate the candidates* cell above; this is the summary. This depth has the **most** device-specific entries of the five, because its probe is the one that failed and was replaced. Events that explain something in the data:

| Data feature | Fieldbook entry |
|---|---|
| 0.1% value steps 2020-04-10 → 2020-04-21, and the two >24 h identical-value runs | 2020-04-21, "New datalogger software with two byte precision for the soil profile measurement (Teros 12 and 21)" — after it, full float precision |
| Failed reads stored as fixed negative numbers rather than gaps | 2020-04-10, the CR1000 firmware predates SDI-12 NaN support and was deliberately not updated |
| Gap 2021-03-24 → 2021-04-02 (in pieces), and the 10MIN → 1MIN change | FF1 logger box rebuilt, all components shut down 24 Mar, new CR1000 and program, data acquisition restarted 26 Mar |
| Gap 2022-09-06 → 2022-09-16 (10 d) | FF1 12 V DC-DC power supply failed; "the data is missing for 10 days ... due to power outage at the FF1", replaced 16 Sep |
| Gap 2023-06-15 (20 h) | FF1 power problem, "No system at ff1 was running for 'one day'"; "FF1 is running again" 16 Jun |
| **Probe silent from 2024-04-11, unplugged 23 Apr 2024** | the 0.3 m probe was failing and "caused an interruption of the SDI12 communication of all sensors at the same port"; unplugged 23 Apr 2024 — but **the data stop 12 days earlier, on 11 Apr**, so the era boundary is dated from the data |
| **New probe from 2025-03-04, 40 cm downslope, ~+3 % VWC level step** | the replacement probe was installed 4 Mar 2025; it could not go in the old hole (the old one could not be removed) so it went 40 cm down the slope — a different soil location, hence the level break |
| Negative reads on all five depths, 18-19 Dec 2025 | two remote logger-program uploads to the FF1 meteo logger on exactly those days |

**Consistent with the other depths but silent here:** the two unexplained station gaps (2024-06-29 → 07-01 and 2024-10-16) and the April-2024 bus interruption all fall **inside** this depth's dead period, so they leave no separate mark in this record — there was no sensor to miss them. The 23 Oct 2025 wind storm (a beech onto the FF1 station) damaged above-ground sensors only; the buried probe logged through it. The 2023-06-30 timezone change applies to the eddy-covariance PC, not the CR1000 that produces this series.

### ♻️ Reusing this notebook for the other depths

This is the `0.3` copy of the shared SWC screening notebook. Its structure is identical to the others; what is **specific to this depth** and must not be carried elsewhere:

1. **The sensor swap** — `SENSOR_SWAP` / `OLD_PROBE_LAST`, the whole *The sensor swap* section, and the per-era splits in the describe/correlation/audit cells exist because this field name spans two probes. Only `0.3` has this. Do not copy the swap machinery to a depth that has a single sensor.
2. **`REMOVE_DATES` is empty** here, and the 5 cm decoupling window was deleted. Re-derive removals from each depth's own evidence; do not carry this one's (empty) list, nor 0.05's window.
3. **The companion mask** (`COMPANION_MASK`) exists to keep 0.05's own 2020 fault out of *this* depth's cross-check. Every depth except 0.05 should keep masking 0.05's window; 0.05 itself needs no such mask.
4. **The absolute-limits count is depth-specific.** `[0, 60]` is physical and holds everywhere, but this depth reaches the test with only 2 records (the old probe went silent rather than logging sentinels); the other depths reach it with more, from the April 2024 bus failure.
5. **If you add any outlier test, report its flagged count per era, never as a total** — and remember this depth has *three* boundaries (resolution at 2021-03-26, sensor at 2024-04-11 and 2025-03-04). This notebook commits no spike test, because after absolute limits the only large excursions left are real wetting fronts.